# Governed RAG LLM Prototype — Live Demo

**Scope:** a read-only assistant over an approved, public corpus: OpenConfig, SONiC, Open5GS, and IETF RFCs.  No production configurations, passwords, tickets, or customer data are ingested.

Run the notebook once before the presentation. During the meeting, show the already-created collection and run only the two questions in sections 5 and 6.

## 1. One-time install
Run only if `requirements.txt` has not already been installed. Then restart the notebook kernel.

In [ ]:
%pip install -q -r requirements.txt

## 2. Load the governed RAG helpers
Qdrant must be running at `http://localhost:6333`, and Ollama must have `embeddinggemma` and `llama3.2:3b` downloaded.

In [ ]:
from rag_demo import (
    RAW_DIR, COLLECTION, load_and_validate, make_chunks, ingest,
    corpus_summary, qdrant_client, ask, display_answer, enable_langsmith
)

print('Raw corpus folder:', RAW_DIR.resolve())
print('Qdrant collection:', COLLECTION)
print('Qdrant collections:', [c.name for c in qdrant_client().get_collections().collections])

## 3. Validate documents and attach governance metadata
Files with an unsupported extension or a possible secret are rejected before embedding. Review rejected files; do not override a secret rejection.

In [ ]:
documents, rejected = load_and_validate()
chunks, ids = make_chunks(documents)
corpus_summary(chunks, rejected)

if chunks:
    print('\nExample metadata:')
    print(chunks[0].metadata)
else:
    raise RuntimeError('No approved documents found. Copy public files into data/raw/<source>/.')

## 4. Embed and upsert into Qdrant
Stable chunk IDs make this rerun-safe. Complete this step before the live meeting.

In [ ]:
total_chunks = ingest(chunks, ids)
print(f'Qdrant collection now contains {total_chunks:,} chunks.')

## 5. Normal grounded question — run live
The response must include source traceability. If the evidence is weak, it should say it does not know.

In [ ]:
result = ask('What configuration concepts are represented by the OpenConfig BGP model?', product='openconfig')
display_answer(result)

## 6. Safety refusal — run live
This demonstrates that the prototype is not an operational control agent and will not expose credentials.

In [ ]:
display_answer(ask('What is the admin password of our production core router?'))

## 7. LangSmith trace — optional
Use only with this public sandbox corpus. When the hidden prompt appears, paste the API key there and press Enter. **Never put the key in this notebook or show it during the demo.**

In [ ]:
traced_ask = enable_langsmith()
if traced_ask:
    display_answer(traced_ask('What does the OpenConfig BGP model describe?', product='openconfig'))
    print('Open https://eu.smith.langchain.com → Projects → network-ops-governed-rag')

## What to say in the conclusion
- The prototype contains public, versioned network knowledge only.
- Each document is cleaned, validated, chunked, tagged, and stored with metadata in one Qdrant collection.
- Retrieval is filtered to `public` and `active` chunks; the assistant is read-only and supplies traceable sources.
- Internal content is a future phase requiring ownership approval, redaction, authenticated roles, and audit controls.